# REPD Tools — Quickstart

This notebook shows the core dataset retrieval and visualisation workflow.

**Data source:** [Renewable Energy Planning Database (REPD)](https://www.gov.uk/government/publications/renewable-energy-planning-database-monthly-extract)  
Download the latest quarterly CSV from DESNZ and update the path below.

In [ ]:
from datetime import datetime
from pathlib import Path

from repd.processor import REPDProcessor
from repd.constants import (
    SUCCESSFUL_DEVELOPMENT_TYPES,
    UNSUCCESSFUL_DEVELOPMENT_TYPES,
    CANCELLED_DEVELOPMENT_TYPES,
    NEUTRAL_DEVELOPMENT_TYPES,
)

REPD_CSV   = Path("../data/REPD_Publication_Q4_2025.csv")
LAUTH_JSON = Path("../data/localauth.json")

## 1. Load and filter the dataset

In [ ]:
processor = REPDProcessor(REPD_CSV)

# Full pipeline: parse dates, filter to records updated since 2025-01-01,
# convert OSGB36 → WGS84, return a GeoDataFrame.
gdf = processor.process_pipeline(date=datetime(2025, 1, 1))
print(f"Loaded {len(gdf):,} projects")
gdf.head(3)

In [ ]:
# Split by outcome
gdf_unsuccessful = processor.filter_by_unsuccessful(gdf)
gdf_successful   = processor.filter_by_successful(gdf)
gdf_cancelled    = processor.filter_by_cancelled(gdf)
gdf_awaiting     = processor.filter_by_status(gdf, ["Awaiting Construction"])
gdf_submitted    = processor.filter_by_status(gdf, ["Application Submitted"])

print(f"Unsuccessful : {len(gdf_unsuccessful):,}")
print(f"Successful   : {len(gdf_successful):,}")
print(f"Cancelled    : {len(gdf_cancelled):,}")
print(f"Awaiting     : {len(gdf_awaiting):,}")
print(f"Submitted    : {len(gdf_submitted):,}")

## 2. Map of unsuccessful projects

In [ ]:
from repd.visualise import plot_status_map

fig = plot_status_map(
    gdf_unsuccessful,
    lauth_path=LAUTH_JSON,
    title="Unsuccessful Renewable Energy Projects (2025+)",
    marker="x",
    size=28,
)
fig.savefig("unsuccessful_projects.png", dpi=150, bbox_inches="tight")

## 3. Outcome summary chart

In [ ]:
from repd.visualise import plot_outcome_summary

# Restrict to records last updated in 2025
fig = plot_outcome_summary(gdf, year=2025)
fig.savefig("outcome_summary_2025.png", dpi=150, bbox_inches="tight")

## 4. Planning delay distributions

In [ ]:
from repd.visualise import plot_delay_distribution

# Planning permission → construction start
fig = plot_delay_distribution(
    gdf,
    from_col="Planning Permission Granted",
    to_col="Under Construction",
    title="Planning Permission → Under Construction",
    bar_color="#c8601e",
)

# Construction start → operational
fig2 = plot_delay_distribution(
    gdf,
    from_col="Under Construction",
    to_col="Operational",
    title="Under Construction → Operational",
    bar_color="#d4b84c",
)

## 5. Choropleth — projects stuck in Awaiting Construction

In [ ]:
from repd.visualise import plot_choropleth

# Load the full unfiltered dataset for maximum geographic coverage
df_full = processor.load()
df_full = processor.convert_datetime(df_full)
df_full_awaiting = processor.filter_by_status(df_full, ["Awaiting Construction"])

AUTHORITY_ALIASES = {
    "Durham": "County Durham",
    "Hyndburn Borough": "Hyndburn",
    "Burnely": "Burnley",
}

fig = plot_choropleth(
    df_full_awaiting,
    lauth_path=LAUTH_JSON,
    col="stuck_count",
    group_col="Planning Authority",
    title="Projects Stuck in Awaiting Construction",
    cmap="Oranges",
    authority_aliases=AUTHORITY_ALIASES,
)
fig.savefig("stuck_choropleth.png", dpi=150, bbox_inches="tight")

## 6. Export to GeoJSON

## 7. Yearly project outcomes — stacked bar chart

For each calendar year, count how many projects reached each status using the event date for that status (not the record update date). This gives a true cohort view of the UK renewable pipeline over time.

In [ ]:
from repd.visualise import plot_yearly_status_stacked

# df_full already loaded in Section 5 (full dataset, dates parsed)
counts = processor.yearly_status_counts(df_full)
print(counts.tail(10))

fig = plot_yearly_status_stacked(
    counts,
    year_range=(2000, 2025),
)
fig.savefig("yearly_status_stacked.png", dpi=150, bbox_inches="tight")

In [ ]:
import json

geojson = processor.create_geojson(gdf_cancelled)
with open("cancelled_projects.geojson", "w") as f:
    json.dump(geojson, f, indent=2)
print(f"Exported {len(geojson['features'])} features")